# Phase 2 — Triage & Profiling Demo

This notebook demonstrates Phase 2: computing cyclomatic complexity and token counts for each function, then applying triage logic.

Functions below the complexity threshold are **triaged out** (skip LLM optimization, pass through unchanged).

In [1]:
# Install dependencies (run once per kernel)
%pip install -q llvmlite tiktoken


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, pathlib
here = pathlib.Path.cwd()
if (here / "llmcompile").exists() and str(here) not in sys.path:
    sys.path.insert(0, str(here))

from llmcompile.phases.p1_parse import parse_module
from llmcompile.phases.p2_triage import triage_module, summarize
from llmcompile.config import PipelineConfig, TriageConfig
print("✓ Imports successful")

✓ Imports successful


## Sample IR with varying complexity

We have functions with different complexity levels:
- `simple_add`: straight-line code (complexity = 1)
- `max`: one conditional (complexity = 2)
- `complex_logic`: nested conditionals (complexity > 2)

In [3]:
SAMPLE_IR = r"""
; ModuleID = 'sample'
source_filename = "sample.c"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
target triple = "x86_64-unknown-linux-gnu"

; Straight-line function: complexity = 1
define i32 @simple_add(i32 %a, i32 %b) {
entry:
  %sum = add i32 %a, %b
  ret i32 %sum
}

; Function with one conditional: complexity = 2
define i32 @max(i32 %a, i32 %b) {
entry:
  %cmp = icmp sgt i32 %a, %b
  br i1 %cmp, label %if.then, label %if.else

if.then:
  ret i32 %a

if.else:
  ret i32 %b
}

; Function with nested conditionals: complexity = 4
define i32 @complex_logic(i32 %x, i32 %y, i32 %z) {
entry:
  %cmp1 = icmp sgt i32 %x, %y
  br i1 %cmp1, label %if.then, label %if.else

if.then:
  %cmp2 = icmp sgt i32 %x, %z
  br i1 %cmp2, label %return.x, label %return.z

if.else:
  %cmp3 = icmp sgt i32 %y, %z
  br i1 %cmp3, label %return.y, label %return.z

return.x:
  ret i32 %x

return.y:
  ret i32 %y

return.z:
  ret i32 %z
}
"""

## Phase 1: Parse & Extract

In [4]:
parsed = parse_module(SAMPLE_IR)
print(f"Extracted {len(parsed.functions)} functions:")
for fn in parsed.functions:
    print(f"  - {fn.name}")

Extracted 3 functions:
  - simple_add
  - max
  - complex_logic


## Phase 2: Triage & Profile

Let's use a threshold of 3 - functions with complexity < 3 will be triaged out.

In [5]:
# Configure triage threshold
config = PipelineConfig(
    triage=TriageConfig(complexity_threshold=3)
)

# Run triage
triage_module(parsed, config)

# Print summary
print(summarize(parsed))

Triage results for 3 function(s):
  - 2 triaged out (too simple)
  - 1 to optimize via LLM

  simple_add               complexity=  1 tokens=  156 [TRIAGED OUT]
  max                      complexity=  2 tokens=  204 [TRIAGED OUT]
  complex_logic            complexity=  4 tokens=  320 [TO OPTIMIZE]


## Detailed Results

Let's examine each function's metrics:

In [6]:
for fn in parsed.functions:
    status = "🔄 TO OPTIMIZE" if not fn.triaged_out else "⏭️  TRIAGED OUT"
    print(f"\n{'='*70}")
    print(f"Function: @{fn.name}")
    print(f"{'='*70}")
    print(f"Cyclomatic Complexity: {fn.complexity}")
    print(f"Token Count:           {fn.token_count}")
    print(f"Status:                {status}")
    print(f"\nRationale:")
    if fn.triaged_out:
        print(f"  → Complexity {fn.complexity} < threshold {config.triage.complexity_threshold}")
        print(f"  → Too simple for LLM optimization, will pass through unchanged")
    else:
        print(f"  → Complexity {fn.complexity} >= threshold {config.triage.complexity_threshold}")
        print(f"  → Will be routed to LLM for optimization in Phase 3")


Function: @simple_add
Cyclomatic Complexity: 1
Token Count:           156
Status:                ⏭️  TRIAGED OUT

Rationale:
  → Complexity 1 < threshold 3
  → Too simple for LLM optimization, will pass through unchanged

Function: @max
Cyclomatic Complexity: 2
Token Count:           204
Status:                ⏭️  TRIAGED OUT

Rationale:
  → Complexity 2 < threshold 3
  → Too simple for LLM optimization, will pass through unchanged

Function: @complex_logic
Cyclomatic Complexity: 4
Token Count:           320
Status:                🔄 TO OPTIMIZE

Rationale:
  → Complexity 4 >= threshold 3
  → Will be routed to LLM for optimization in Phase 3


## Understanding Cyclomatic Complexity

Cyclomatic complexity measures the number of independent paths through code:

- **Complexity = 1**: Straight-line code with no branches
- **Complexity = 2**: One decision point (if/else)
- **Complexity = N**: N-1 decision points

Each conditional branch, switch case, or loop adds to complexity.

## Experiment: Different Thresholds

Let's see how different thresholds affect triage decisions:

In [7]:
for threshold in [1, 2, 3, 5, 10]:
    parsed_test = parse_module(SAMPLE_IR)
    config_test = PipelineConfig(triage=TriageConfig(complexity_threshold=threshold))
    triage_module(parsed_test, config_test)
    
    triaged = sum(1 for f in parsed_test.functions if f.triaged_out)
    to_opt = len(parsed_test.functions) - triaged
    
    print(f"Threshold={threshold:2d}: {triaged} triaged out, {to_opt} to optimize")

Threshold= 1: 0 triaged out, 3 to optimize
Threshold= 2: 1 triaged out, 2 to optimize
Threshold= 3: 2 triaged out, 1 to optimize
Threshold= 5: 3 triaged out, 0 to optimize
Threshold=10: 3 triaged out, 0 to optimize


## Token Routing Preview

Token counts will determine which LLM tier handles each function in Phase 3:

In [8]:
print("\nPhase 3 routing preview (based on token count):\n")
for fn in parsed.functions:
    if fn.triaged_out:
        tier = "(skipped - triaged out)"
    elif fn.token_count < 8000:
        tier = "fast (local models)"
    elif fn.token_count < 32000:
        tier = "mid (Llama 8B, Qwen 7B)"
    else:
        tier = "frontier (GPT-4o, Claude)"
    
    print(f"  {fn.name:<20} {fn.token_count:>5} tokens → {tier}")


Phase 3 routing preview (based on token count):

  simple_add             156 tokens → (skipped - triaged out)
  max                    204 tokens → (skipped - triaged out)
  complex_logic          320 tokens → fast (local models)


## Next Steps

Phase 2 is now complete! The pipeline can:
1. ✅ Parse IR and extract functions (Phase 1)
2. ✅ Compute complexity and token metrics (Phase 2)
3. ✅ Triage simple functions to skip optimization

Next to build:
- **Phase 5**: Verification gate (llvm-as + Alive2)
- **Phase 4 & 6**: Reconstruction and assembly
- **Orchestrator**: Tie everything together with identity transform
- **Phase 3**: LLM routing (last, after verification is proven)